<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_02_video_frame_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 2: Video Inventory, Manifest Construction, and Frame Validation

This notebook inventories the physical Kvasir-Capsule videos in persistent storage and links them to the validated metadata produced in Sector 1. It checks file availability and identifier uniqueness, reads technical video properties, and builds a video-level manifest containing file paths, sizes, annotation status, and container metadata.

Current checks cover container opening, first-frame readability, and consistency between the manifest and annotation metadata. Full sequential decoding is being added to count successfully decoded frames, record decoding outcomes, and investigate differences between observed counts and published dataset references.

Video manifests and audit results are saved to persistent storage to support downstream frame extraction, temporal processing, and dataset preparation.

### 1. Install dependencies and imports

In [6]:
%pip install -q \
    pandas \
    numpy \
    pyarrow \
    opencv-python-headless \
    scikit-image \
    scikit-learn \
    iterative-stratification \
    tqdm \
    gdown

In [7]:
from pathlib import Path
from collections import defaultdict

import re
import mimetypes
import os
import json
import random
import warnings
import subprocess
import shutil
import sys
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from IPython.display import display
from collections import deque
from uuid import uuid4
from contextlib import contextmanager
from collections import Counter
from itertools import islice


import filecmp
import lzma
import stat
import tempfile
import cv2
import numpy as np
import pandas as pd
import torch
import zipfile
import tarfile
import zlib
import gzip
import ast
import hashlib


from tqdm.auto import tqdm

from skimage.metrics import structural_similarity as ssim

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

### 2. Github Colab Sync

In [8]:
GIT_CONFIG = {
    "repo_url": "https://github.com/bogdanparvu18/msc-graduate-project.git",
    "branch": "main",
    "local_repo_dir": "/content/msc-graduate-project",
    "config_notebook_name": "phase2_01_ingestion.ipynb",
    "repo_output_dir": "outputs/phase2",
    "config_output_filename": "phase2_01_ingestion_{run_id}_config.json",
    "output_groups": ["configs", "results", "reports"],
    "dataset_audit_subdir": "dataset_audit",
    "dataset_audit_excluded_artifacts": [
        "physical_image_inventory",
        "metadata_clean",
    ],
    "allowed_extensions": [".json", ".csv", ".md", ".txt"],
    "max_file_size_mib": 20,
    "token_secret_name": "GITHUB_TOKEN",
    "github_username": "bogdanparvu18",
    "author_name": "Bogdan Parvu",
    "author_email": "bparvu@lakeheadu.ca",
    "commit_message": "Phase 2: update ingestion config and audit outputs",
    "confirm_push": True,
}


def run_git(repo, *args, env=None, input_text=None, allowed_codes=(0,)):
    """Runs Git without a shell; never embeds credentials in arguments."""
    command = ["git"] + (["-C", str(repo)] if repo is not None else [])
    process_env = {
        k: v for k, v in os.environ.items()
        if not k.startswith("GIT_TRACE") and k != "GIT_CURL_VERBOSE"
    }
    process_env.update({"GIT_TERMINAL_PROMPT": "0", **(env or {})})
    result = subprocess.run(
        command + list(args), input=input_text, env=process_env,
        text=True, capture_output=True,
    )
    if result.returncode not in allowed_codes:
        message = (result.stderr or result.stdout).strip()
        token = process_env.get("GITHUB_TOKEN")
        if token:
            message = message.replace(token, "[REDACTED]")
        raise RuntimeError(f"Git failed:\n{message}")
    return result.stdout


def prepare_git_repository(settings):
    """Clones once or fast-forwards a clean, matching local repository."""
    repo = Path(settings["local_repo_dir"]).expanduser().resolve()
    branch = settings["branch"]
    if not (repo / ".git").is_dir():
        if repo.exists() and (not repo.is_dir() or any(repo.iterdir())):
            raise RuntimeError(f"Destination exists but is not an empty Git checkout: {repo}")
        repo.parent.mkdir(parents=True, exist_ok=True)
        run_git(None, "clone", "--branch", branch, "--single-branch",
                settings["repo_url"], str(repo))

    expected = settings["repo_url"].rstrip("/").removesuffix(".git")
    for options in [("get-url",), ("get-url", "--push")]:
        actual = run_git(repo, "remote", *options, "origin").strip()
        if actual.rstrip("/").removesuffix(".git") != expected:
            raise RuntimeError("Origin does not match GIT_CONFIG. Nothing was pushed.")
    if run_git(repo, "branch", "--show-current").strip() != branch:
        raise RuntimeError(f"Expected branch {branch}; no automatic branch switching.")
    if run_git(repo, "status", "--porcelain").strip():
        raise RuntimeError("Local checkout has changes. Resolve them before synchronizing; no reset is performed.")

    run_git(repo, "pull", "--ff-only", "origin", branch)
    if run_git(repo, "rev-list", f"origin/{branch}..HEAD").strip():
        raise RuntimeError("There are unpublished local commits. Resolve/push them before starting another sync.")
    return repo


def load_config_from_notebook(repo, notebook_name):
    """Reads the literal CONFIG dictionary; does NOT execute the notebook."""
    tracked = run_git(repo, "ls-files", "-z").split("\0")
    matches = [repo / p for p in tracked if p and Path(p).name == notebook_name]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one tracked {notebook_name}; found: {matches}")
    notebook_path = matches[0]
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    configs = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        source = "".join(source) if isinstance(source, list) else source
        try:
            statements = ast.parse(source).body
        except SyntaxError:
            if re.search(r"(?m)^\s*CONFIG\s*=", source):
                raise ValueError("Keep CONFIG = {...} in a plain Python cell without shell/magic commands.") from None
            continue
        for node in statements:
            targets = node.targets if isinstance(node, ast.Assign) else (
                [node.target] if isinstance(node, ast.AnnAssign) else []
            )
            if not any(isinstance(t, ast.Name) and t.id == "CONFIG" for t in targets):
                continue
            if not isinstance(node.value, ast.Dict):
                continue  # Ignore calls such as CONFIG = load_config(...).
            try:
                configs.append(ast.literal_eval(node.value))
            except (ValueError, TypeError, SyntaxError):
                raise ValueError("CONFIG must contain literal values, not calls, external variables, or **CONFIG.") from None
    if len(configs) != 1:
        raise ValueError(f"Expected one literal CONFIG dictionary; found {len(configs)}.")
    json.dumps(configs[0], allow_nan=False)  # Check JSON compatibility.
    return configs[0], notebook_path


def file_sha256(path):
    """Hashes file contents without loading the entire file into RAM."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def collect_publishable_outputs(dirs, repo, settings):
    """Builds a publication plan from output directories only."""
    relative_root = Path(settings["repo_output_dir"])
    if relative_root.is_absolute() or ".." in relative_root.parts:
        raise ValueError("repo_output_dir must be a repository-relative path.")
    records, skipped = [], []
    for group in settings["output_groups"]:
        if group not in {"configs", "results", "reports"}:
            raise ValueError(f"Unsupported output group: {group}")
        source_dir = Path(dirs[f"{group}_dir"]).resolve()
        if not source_dir.is_dir():
            raise FileNotFoundError(f"Output directory is missing: {source_dir}")
        for source in sorted(source_dir.rglob("*")):
            if not source.is_file():
                continue
            relative = source.relative_to(source_dir)
            if any(part.startswith(".") for part in relative.parts) or source.suffix.lower() not in settings["allowed_extensions"]:
                skipped.append(f"{group}/{relative.as_posix()}")
                continue
            if source.is_symlink() or not source.resolve().is_relative_to(source_dir):
                raise ValueError(f"Refusing a symlink/out-of-scope source: {source}")
            size = source.stat().st_size
            if size > settings["max_file_size_mib"] * 1024 ** 2:
                raise ValueError(f"File exceeds publication size limit: {source}")
            repo_relative = relative_root / group / relative
            destination = repo / repo_relative
            if destination.is_symlink() or not destination.resolve().is_relative_to(repo):
                raise ValueError(f"Unsafe destination: {destination}")
            digest = file_sha256(source)
            records.append({
                "source": source, "destination": destination,
                "repo_relative": repo_relative.as_posix(), "group": group,
                "size": size, "sha256": digest,
                "changed": not destination.exists() or file_sha256(destination) != digest,
            })
    return records, skipped


def build_dataset_audit_github_files(
    storage_spec, catalog_filename, excluded_artifacts,
):
    """Derives audit filenames from the specification used by the exporter."""
    missing_columns = sorted({"artifact", "format"} - set(storage_spec.columns))
    if missing_columns:
        raise KeyError(f"Audit storage specification is missing: {missing_columns}")

    selected = (
        storage_spec.loc[
            ~storage_spec["artifact"].isin(excluded_artifacts),
            ["artifact", "format"],
        ]
        .astype("string")
        .assign(filename=lambda data: data["artifact"] + "." + data["format"])
    )
    # The exporter writes this catalog separately from AUDIT_STORAGE_SPEC.
    filenames = pd.concat(
        [selected["filename"], pd.Series([catalog_filename], dtype="string")],
        ignore_index=True,
    )
    valid = filenames.str.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_.-]*", na=False)
    if not valid.all():
        raise ValueError(
            "Audit exports must have nonempty, simple filenames: "
            f"{filenames.loc[~valid].tolist()}"
        )
    return tuple(filenames.drop_duplicates().sort_values())


def collect_publishable_dataset_audit(
    dirs, repo, settings, storage_spec, catalog_filename,
):
    """Plans audit publication; existing remote files are never updated.

    Call after prepare_git_repository(), so HEAD matches the remote branch.
    This function inspects files but does not copy, stage, commit or push.
    """
    subdir = Path(settings["dataset_audit_subdir"])
    if subdir.is_absolute() or ".." in subdir.parts or subdir == Path("."):
        raise ValueError("dataset_audit_subdir must be a nonempty relative path.")
    relative_root = Path(settings["repo_output_dir"])
    if relative_root.is_absolute() or ".." in relative_root.parts:
        raise ValueError("repo_output_dir must be a repository-relative path.")

    source_dir = (Path(dirs["curated_data_dir"]) / subdir).resolve()
    destination_root = relative_root / subdir
    excluded_artifacts = settings["dataset_audit_excluded_artifacts"]
    filenames = build_dataset_audit_github_files(
        storage_spec, catalog_filename, excluded_artifacts,
    )
    excluded_spec = storage_spec.loc[
        storage_spec["artifact"].isin(excluded_artifacts), ["artifact", "format"]
    ].astype("string")
    skipped = [
        f"{subdir.as_posix()}/{name}"
        for name in (excluded_spec["artifact"] + "." + excluded_spec["format"])
    ]
    allowed_extensions = {extension.lower() for extension in settings["allowed_extensions"]}
    tracked_files = set(filter(None, run_git(
        repo, "--literal-pathspecs", "ls-tree", "-r", "--name-only", "-z",
        "HEAD", "--", destination_root.as_posix(),
    ).split("\0")))

    records = []
    for filename in filenames:
        if Path(filename).suffix.lower() not in allowed_extensions:
            skipped.append(f"{subdir.as_posix()}/{filename}")
            continue
        source = source_dir / filename
        repo_relative = (destination_root / filename).as_posix()
        destination = repo / repo_relative
        if destination.is_symlink() or not destination.resolve().is_relative_to(repo):
            raise ValueError(f"Unsafe destination: {destination}")

        already_on_github = repo_relative in tracked_files
        size, digest = None, None
        if not already_on_github:
            if not source.is_file():
                raise FileNotFoundError(
                    f"Audit report is absent from GitHub and Drive: {source}. "
                    "Run the dataset audit export cell first."
                )
            if source.is_symlink() or not source.resolve().is_relative_to(source_dir):
                raise ValueError(f"Refusing a symlink/out-of-scope source: {source}")
            size = source.stat().st_size
            if size > settings["max_file_size_mib"] * 1024 ** 2:
                raise ValueError(f"File exceeds publication size limit: {source}")
            digest = file_sha256(source)

        records.append({
            "source": source, "destination": destination,
            "repo_relative": repo_relative, "group": "dataset_audit",
            "size": size, "sha256": digest,
            "changed": not already_on_github,
        })
    return records, skipped


def authenticated_push(repo, settings, token):
    """Passes the token via an ephemeral environment, not the remote URL."""
    with tempfile.TemporaryDirectory(prefix="mmvqa-git-") as folder:
        askpass = Path(folder) / "askpass.sh"
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n'
            '*Username*) printf "%s\\n" "$MMVQA_GIT_USER" ;;\n'
            '*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            'esac\n', encoding="utf-8",
        )
        askpass.chmod(0o700)
        environment = {
            "GIT_ASKPASS": str(askpass),
            "MMVQA_GIT_USER": settings["github_username"],
            "GITHUB_TOKEN": token,
            "LC_ALL": "C",
        }
        run_git(repo, "-c", "credential.helper=", "push", "origin",
                f"HEAD:refs/heads/{settings['branch']}", env=environment)


def publish_phase2_outputs(config, dirs, settings):
    """Publishes runtime CONFIG, saved outputs and missing dataset audit files.

    Regular outputs are updated when their content changes. Audit filenames
    follow AUDIT_STORAGE_SPEC; audit files already on GitHub are preserved.
    All selected changes share the existing confirmation, commit and push.
    """
    if config["storage_backend"] == "google_drive" and not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Mount Google Drive before publishing.")
    expected_root = Path(config["output_dir"])
    if not expected_root.is_absolute():
        expected_root = Path(config["storage_root"]) / expected_root
    expected_root = expected_root.resolve()
    for group in settings["output_groups"]:
        if Path(dirs[f"{group}_dir"]).resolve() != expected_root / group:
            raise ValueError("DIRS and CONFIG point to different output locations.")
        if not Path(dirs[f"{group}_dir"]).is_dir():
            raise FileNotFoundError(f"Run prepare_phase2_dirs first: {group}")

    # Audit definitions are needed only here, not during the bootstrap below.
    missing_audit_definitions = [
        name for name in ("AUDIT_STORAGE_SPEC", "AUDIT_CATALOG_FILENAME")
        if name not in globals()
    ]
    if missing_audit_definitions:
        raise RuntimeError(
            "Run the dataset audit definition/export cell before publishing. "
            f"Missing: {missing_audit_definitions}"
        )

    repo = prepare_git_repository(settings)  # Refresh before copying, not afterwards.
    run_id = config.get("run_id")
    if not isinstance(run_id, str) or not run_id.strip():
        raise ValueError("Initialize CONFIG['run_id'] at the start of the run before publishing.")
    config_filename = settings["config_output_filename"].format(run_id=run_id)
    if Path(config_filename).name != config_filename or not config_filename.endswith(".json"):
        raise ValueError("config_output_filename must resolve to a JSON filename, not a path.")
    config_path = Path(dirs["configs_dir"]) / config_filename
    config_text = json.dumps(config, indent=2, ensure_ascii=False, sort_keys=True, allow_nan=False) + "\n"
    config_path.write_text(config_text, encoding="utf-8")
    records, skipped = collect_publishable_outputs(dirs, repo, settings)
    audit_records, audit_skipped = collect_publishable_dataset_audit(
        dirs=dirs, repo=repo, settings=settings,
        storage_spec=AUDIT_STORAGE_SPEC,
        catalog_filename=AUDIT_CATALOG_FILENAME,
    )
    records = [*records, *audit_records]
    skipped = [*skipped, *audit_skipped]
    planned_paths = pd.Series([r["repo_relative"] for r in records], dtype="string")
    if planned_paths.duplicated().any():
        raise ValueError("Multiple publication sources target the same repository path.")
    if not any(r["group"] in {"reports", "results", "dataset_audit"} for r in records):
        raise RuntimeError("No publishable reports/results found. Save them to Drive first.")
    audit_publication = {
        "already_on_github": [r["repo_relative"] for r in audit_records if not r["changed"]],
        "published": [],
    }
    changed = [r for r in records if r["changed"]]
    print(f"Eligible: {len(records)} | Changed: {len(changed)} | Excluded: {len(skipped)}")
    print(f"Dataset audit already on GitHub: {len(audit_publication['already_on_github'])}")
    for path in audit_publication["already_on_github"]:
        print("ALREADY ON GITHUB:", path)
    for path in skipped:
        print("EXCLUDED:", path)
    if not changed:
        print("No files need publication under the configured policies. No commit or push is needed.")
        return {"status": "unchanged", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}

    paths = [r["repo_relative"] for r in changed]
    path_input = "\0".join(paths) + "\0"
    ignored = run_git(repo, "check-ignore", "--stdin", "-z",
                      input_text=path_input, allowed_codes=(0, 1))
    if ignored:
        raise RuntimeError("Adjust .gitignore for these outputs first: " + ", ".join(filter(None, ignored.split("\0"))))
    for record in changed:
        print(f"PUBLISH: {record['repo_relative']} ({record['size'] / 1024:.1f} KiB)")
    if settings["confirm_push"] and input("Publish these files? Type PUSH: ").strip() != "PUSH":
        return {"status": "cancelled", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}

    from google.colab import userdata
    token = userdata.get(settings["token_secret_name"]).strip()
    if not token:
        raise ValueError("The GitHub token is empty.")
    email = settings["author_email"].strip() or input("Git commit email (GitHub/noreply): ").strip()
    if "@" not in email:
        raise ValueError("Provide your GitHub commit email or your exact GitHub noreply address.")
    # Detect changes made while the user was reviewing the publication plan.
    if any(file_sha256(r["source"]) != r["sha256"] for r in changed):
        raise RuntimeError("An output changed during review. Run publication again.")
    if any(token.encode() in r["source"].read_bytes() for r in changed):
        raise ValueError("An output contains the GitHub token. Publication stopped.")
    for record in changed:
        record["destination"].parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(record["source"], record["destination"])
    run_git(repo, "--literal-pathspecs", "add", "--pathspec-from-file=-",
            "--pathspec-file-nul", input_text=path_input)
    staged = set(filter(None, run_git(repo, "diff", "--cached", "--name-only", "-z").split("\0")))
    if staged - set(paths):
        raise RuntimeError("Unexpected staged files. Inspect the local checkout before proceeding.")
    if not staged:
        print("No changes remain after Git text normalization.")
        return {"status": "unchanged", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}
    print(run_git(repo, "diff", "--cached", "--stat"))
    run_git(repo, "-c", f"user.name={settings['author_name']}", "-c", f"user.email={email}",
            "commit", "-m", settings["commit_message"])
    authenticated_push(repo, settings, token)
    commit = run_git(repo, "rev-parse", "HEAD").strip()
    audit_publication["published"] = sorted(
        staged & {r["repo_relative"] for r in audit_records}
    )
    print(f"Pushed {len(staged)} files to {settings['branch']}. Commit: {commit}")
    return {"status": "pushed", "published": len(staged), "commit": commit,
            "excluded": skipped, "dataset_audit": audit_publication}


# Bootstrap only: no dataset download and no report generation.
REPO_DIR = prepare_git_repository(GIT_CONFIG)

CONFIG, CONFIG_NOTEBOOK_PATH = load_config_from_notebook(
    repo=REPO_DIR,
    notebook_name=GIT_CONFIG["config_notebook_name"],
)

# Initialize ONCE, before running the pipeline, not inside publication.
# Literal defaults maintain compatibility with existing repository CONFIGs.
# Defining these two keys in the source CONFIG overrides these defaults.
CONFIG.setdefault("timezone", "America/Toronto")
CONFIG.setdefault("run_id_format", "%Y%m%d_%H%M%S")

RUN_ID = datetime.now(
    ZoneInfo(CONFIG["timezone"])
).strftime(CONFIG["run_id_format"])

CONFIG["run_id"] = RUN_ID

CONFIG_SOURCE_COMMIT = run_git(REPO_DIR, "rev-parse", "HEAD").strip()

print("Repository:", REPO_DIR)
print("CONFIG loaded from:", CONFIG_NOTEBOOK_PATH.relative_to(REPO_DIR))
print("Source commit:", CONFIG_SOURCE_COMMIT)
print("Run ID:", CONFIG["run_id"])
print("Run the existing DIRS, ingestion, validation, and report-saving cells next.")

Repository: /content/msc-graduate-project
CONFIG loaded from: notebooks/phase2/phase2_01_ingestion.ipynb
Source commit: 8f60fc7d05a45af3795a63cad12c52b363d91c21
Run ID: 20260924_121822
Run the existing DIRS, ingestion, validation, and report-saving cells next.


### 3. Load declarative configuration and reproducibility

In [28]:
CONFIG = {
    # --------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------

    "seed": 42,

    # --------------------------------------------------------------
    # Persistent storage
    # --------------------------------------------------------------

    "storage_backend": "google_drive",

    "storage_root": (
        "/content/drive/MyDrive/"
        "MMVQA_Clinical"
    ),

    "raw_data_dir": (
        "data/raw/kvasir_capsule"
    ),

    "interim_data_dir": (
        "data/interim/phase2"
    ),

    "curated_data_dir": (
        "data/curated/phase2"
    ),

    "output_dir": (
        "outputs/phase2"
    ),

    # --------------------------------------------------------------
    # Raw-dataset acquisition
    # --------------------------------------------------------------

    "dataset_download_enabled": True,

    "dataset_google_drive_folder_url": (
        "https://drive.google.com/drive/folders/"
        "18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z"
    ),

    "dataset_archive_repair_enabled": True,

    # --------------------------------------------------------------
    # Raw-dataset validation
    # --------------------------------------------------------------

    "dataset_validation": {
        "osf_storage_subdir": (
            "osfstorage"
        ),

        "required_metadata_file": (
            "metadata.csv"
        ),

        "labelled_images_subdir": (
            "labelled_images"
        ),

        "minimum_video_files": 117,

        "minimum_labelled_images": 47238,

        "image_extensions": [
            ".png",
            ".jpg",
            ".jpeg",
        ],

        "video_extensions": [
            ".avi",
            ".mp4",
            ".mkv",
        ],
    },


    # --------------------------------------------------------------
    # Normalized clinical taxonomy
    # --------------------------------------------------------------

    # Keys must match finding_class_normalized.
    "clinical_group_map": {
        "ampulla of vater": (
            "anatomical_landmark"
        ),

        "ileocecal valve": (
            "anatomical_landmark"
        ),

        "pylorus": (
            "anatomical_landmark"
        ),

        "normal clean mucosa": (
            "normal_mucosa"
        ),

        "reduced mucosal view": (
            "visibility_limitation"
        ),

        "blood fresh": "bleeding",

        "blood hematin": "bleeding",

        "angiectasia": (
            "vascular_lesion"
        ),

        "erosion": (
            "mucosal_lesion"
        ),

        "erythema": (
            "mucosal_lesion"
        ),

        "ulcer": (
            "mucosal_lesion"
        ),

        "lymphangiectasia": (
            "lymphatic_lesion"
        ),

        "polyp": (
            "protruding_lesion"
        ),

        "foreign body": (
            "foreign_body"
        ),
    },

    # --------------------------------------------------------------
    # Video and frame alignment
    # --------------------------------------------------------------


        "expected_export_container_fps": 30.0,

        "frame_index_offset_candidates": [
            -1,
            0,
            1,
        ],

        "frame_alignment_sample_size": 30,
        "frame_alignment_min_valid_fraction": 0.80,

        "frame_alignment_comparison_size": [
            128,
            128,
        ],

    # --------------------------------------------------------------
    # Expected dataset characteristics
    # --------------------------------------------------------------

        "expected_labelled_frames": 47238,
        "expected_classes": 14,

        "expected_labelled_videos": 43,

        "expected_unlabelled_videos": 74,

        "expected_total_videos": 117,

        "expected_total_extractable_frames": (
            4741504
        ),

        "expected_unlabelled_frames": (
            4694266
        ),
}


CONFIG

{'seed': 42,
 'storage_backend': 'google_drive',
 'storage_root': '/content/drive/MyDrive/MMVQA_Clinical',
 'raw_data_dir': 'data/raw/kvasir_capsule',
 'interim_data_dir': 'data/interim/phase2',
 'curated_data_dir': 'data/curated/phase2',
 'output_dir': 'outputs/phase2',
 'dataset_download_enabled': True,
 'dataset_google_drive_folder_url': 'https://drive.google.com/drive/folders/18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z',
 'dataset_archive_repair_enabled': True,
 'dataset_validation': {'osf_storage_subdir': 'osfstorage',
  'required_metadata_file': 'metadata.csv',
  'labelled_images_subdir': 'labelled_images',
  'minimum_video_files': 117,
  'minimum_labelled_images': 47238,
  'image_extensions': ['.png', '.jpg', '.jpeg'],
  'video_extensions': ['.avi', '.mp4', '.mkv']},
 'clinical_group_map': {'ampulla of vater': 'anatomical_landmark',
  'ileocecal valve': 'anatomical_landmark',
  'pylorus': 'anatomical_landmark',
  'normal clean mucosa': 'normal_mucosa',
  'reduced mucosal view': 'visibilit

### 4. Mount Google Drive Storage Backend

In [10]:
def mount_storage(config):
    """
    Mounts persistent storage when required by the configured backend.

    For Google Colab + Google Drive, this mounts Drive under /content/drive.
    It does not download or copy the dataset.
    """

    storage_backend = config["storage_backend"]

    if storage_backend == "google_drive":

        try:
            from google.colab import drive

            drive.mount(
                "/content/drive",
                force_remount=False,
            )

            print("Google Drive mounted.")

        except ImportError:
            raise RuntimeError(
                "Google Drive backend is configured, "
                "but the notebook is not running in Google Colab."
            )

    elif storage_backend == "local":

        print("Using local storage.")

    else:

        raise ValueError(
            f"Unsupported storage backend: {storage_backend}"
        )


mount_storage(CONFIG)

Mounted at /content/drive
Google Drive mounted.


### 5. Define data paths

In [11]:
def prepare_phase2_dirs(config):
    """
    Resolves and creates the directory structure required
    for Phase 2.

    Main directories are declared in CONFIG. Relative paths
    are resolved against storage_root, while absolute paths
    are preserved.

    Raw-dataset reference paths are returned, but dataset
    source files and source subdirectories are not created.

    Side effect:
        Creates writable Phase 2 directories on persistent
        storage.

    Returns:
        Dictionary containing resolved Path objects.
    """

    storage_root = Path(
        config["storage_root"]
    )

    validation = config[
        "dataset_validation"
    ]


    def resolve_path(path_value):
        """
        Resolves a CONFIG path against storage_root.

        Absolute paths are returned unchanged.
        """

        path = Path(path_value)

        if path.is_absolute():
            return path

        return storage_root / path


    # --------------------------------------------------------------
    # Main CONFIG directories
    # --------------------------------------------------------------

    raw_data_dir = resolve_path(
        config["raw_data_dir"]
    )

    interim_data_dir = resolve_path(
        config["interim_data_dir"]
    )

    curated_data_dir = resolve_path(
        config["curated_data_dir"]
    )

    output_dir = resolve_path(
        config["output_dir"]
    )


    # --------------------------------------------------------------
    # Raw-dataset source references
    # --------------------------------------------------------------

    dataset_root_dir = (
        raw_data_dir
        / validation["osf_storage_subdir"]
    )

    labelled_images_dir = (
        dataset_root_dir
        / validation["labelled_images_subdir"]
    )

    metadata_path = (
        dataset_root_dir
        / validation["required_metadata_file"]
    )


    # --------------------------------------------------------------
    # Writable directories created by the pipeline
    # --------------------------------------------------------------

    created_directories = {
        # Main directories
        "raw_data_dir":
            raw_data_dir,

        "interim_data_dir":
            interim_data_dir,

        "curated_data_dir":
            curated_data_dir,

        "output_dir":
            output_dir,

        # Derived data directories
        "temporal_frames_dir":
            interim_data_dir
            / "temporal_frames",

        "manifests_dir":
            curated_data_dir
            / "manifests",

        "splits_dir":
            curated_data_dir
            / "splits",

        # Derived output directories
        "configs_dir":
            output_dir
            / "configs",

        "results_dir":
            output_dir
            / "results",

        "reports_dir":
            output_dir
            / "reports",
    }


    # --------------------------------------------------------------
    # Create only writable pipeline directories
    # --------------------------------------------------------------

    for directory in created_directories.values():
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )


    # --------------------------------------------------------------
    # Dataset paths that must come from the OSF dataset
    # --------------------------------------------------------------

    dataset_paths = {
        "dataset_root_dir":
            dataset_root_dir,

        "labelled_images_dir":
            labelled_images_dir,

        "metadata_path":
            metadata_path,
    }


    return {
        **created_directories,
        **dataset_paths,
    }


DIRS = prepare_phase2_dirs(
    CONFIG
)

DIRS

{'raw_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/raw/kvasir_capsule'),
 'interim_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/interim/phase2'),
 'curated_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2'),
 'output_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2'),
 'temporal_frames_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/interim/phase2/temporal_frames'),
 'manifests_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/manifests'),
 'splits_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/splits'),
 'configs_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/configs'),
 'results_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/results'),
 'reports_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/reports'),
 'dataset_root_dir': PosixPath('/content/drive/MyDrive/MMVQA

### 6. Verify input paths and inspect video files and metadata columns

In [12]:
root = DIRS["dataset_root_dir"]
metadata_path = DIRS["metadata_path"]

if not root.is_dir():
    raise FileNotFoundError(f"Dataset root is missing: {root}")

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata is missing: {metadata_path}")

print("Dataset root:", root)
print("\nTop-level entries:")
for path in islice(root.iterdir(), 30):
    print("DIR " if path.is_dir() else "FILE", path.name)

# Sector 1 reads this metadata as semicolon-delimited.
metadata_preview = pd.read_csv(metadata_path, sep=";", nrows=30)
print("\nMetadata columns:", metadata_preview.columns.tolist())
display(metadata_preview)

video_extensions = {
    ext.lower()
    for ext in CONFIG["dataset_validation"]["video_extensions"]
}
video_files = sorted(
    path for path in root.rglob("*")
    if path.is_file() and path.suffix.lower() in video_extensions
)

print("\nVideos found:", len(video_files))
print("Video directories:")
for directory, count in Counter(
    str(path.parent.relative_to(root)) for path in video_files
).most_common(10):
    print(f"  {directory}: {count}")

print("\nSample video paths:")
for path in video_files[:10]:
    print(" ", path.relative_to(root))

Dataset root: /content/drive/MyDrive/MMVQA_Clinical/data/raw/kvasir_capsule/osfstorage

Top-level entries:
FILE metadata.json
FILE metadata.csv
DIR  labelled_images
DIR  labelled_videos
DIR  unlabelled_videos

Metadata columns: ['filename', 'video_id', 'frame_number', 'finding_category', 'finding_class', 'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4']


,filename,video_id,frame_number,finding_category,finding_class,x1,y1,x2,y2,x3,y3,x4,y4
0,0728084c8da942d9_22803.jpg,0728084c8da942d9,22803,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0728084c8da942d9_22804.jpg,0728084c8da942d9,22804,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0728084c8da942d9_22805.jpg,0728084c8da942d9,22805,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0728084c8da942d9_22806.jpg,0728084c8da942d9,22806,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0728084c8da942d9_22807.jpg,0728084c8da942d9,22807,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0728084c8da942d9_22808.jpg,0728084c8da942d9,22808,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0728084c8da942d9_22809.jpg,0728084c8da942d9,22809,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0728084c8da942d9_22810.jpg,0728084c8da942d9,22810,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0728084c8da942d9_22811.jpg,0728084c8da942d9,22811,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0728084c8da942d9_22812.jpg,0728084c8da942d9,22812,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Videos found: 117
Video directories:
  unlabelled_videos: 74
  labelled_videos: 43

Sample video paths:
  labelled_videos/04a78ef00c5245e0.mp4
  labelled_videos/0531325b64674948.mp4
  labelled_videos/0728084c8da942d9.mp4
  labelled_videos/07c1fa15a20a4398.mp4
  labelled_videos/131368cc17e44240.mp4
  labelled_videos/2fc3db471f9d44c0.mp4
  labelled_videos/39960e5e099a45ca.mp4
  labelled_videos/3ada4222967f421d.mp4
  labelled_videos/3c8d5f0b90d7475d.mp4
  labelled_videos/4560e83f9afc4685.mp4


### 7. Load validated metadata from Sector 1

In [13]:
metadata_path = (
    DIRS["curated_data_dir"]
    / "dataset_audit"
    / "metadata_clean.parquet"
)

if not metadata_path.is_file():
    raise FileNotFoundError(metadata_path)

df_clean = pd.read_parquet(metadata_path)
display(df_clean[["video_id", "video_key", "frame_number"]].head())

,video_id,video_key,frame_number
0,0728084c8da942d9,0728084c8da942d9,22803
1,0728084c8da942d9,0728084c8da942d9,22804
2,0728084c8da942d9,0728084c8da942d9,22805
3,0728084c8da942d9,0728084c8da942d9,22806
4,0728084c8da942d9,0728084c8da942d9,22807


8. Match annotated video IDs to physical video files

In [14]:
video_index = {}

for path in video_files:
    key = path.stem

    if key in video_index:
        raise ValueError(f"Duplicate video filename stem: {key}")

    video_index[key] = path

annotated_keys = set(df_clean["video_key"].dropna())
missing_keys = annotated_keys - set(video_index)

video_inventory = pd.DataFrame([
    {
        "video_id_from_filename": path.stem,
        "relative_path": path.relative_to(root).as_posix(),
        "filename": path.name,
        "size_bytes": path.stat().st_size,
    }
    for path in video_files
])

minimum = CONFIG["dataset_validation"]["minimum_video_files"]
if len(video_inventory) < minimum:
    raise ValueError(
        f"Found {len(video_inventory)} videos; expected at least {minimum}"
    )

inventory_path = DIRS["manifests_dir"] / "video_inventory.csv"
video_inventory.to_csv(inventory_path, index=False)

print(f"Saved {len(video_inventory)} videos to {inventory_path}")

print("Annotated video IDs:", len(annotated_keys))
print("IDs without a matching file:", len(missing_keys))
print("Examples:", sorted(missing_keys)[:10])

Saved 117 videos to /content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/manifests/video_inventory.csv
Annotated video IDs: 43
IDs without a matching file: 0
Examples: []


### 9. Build physical video manifest

In [15]:
# ------------------------------------------------------------------
# Probe video metadata
# ------------------------------------------------------------------

VIDEO_PROBE_COLUMNS = [
    "container_opened",
    "first_frame_readable",
    "container_fps",
    "container_reported_frame_count",
    "width",
    "height",
    "estimated_container_duration_seconds",
]


def empty_video_probe():
    """
    Returns the stable technical schema for an
    unreadable video container.
    """

    return {
        "container_opened":
            False,

        "first_frame_readable":
            False,

        "container_fps":
            np.nan,

        "container_reported_frame_count":
            np.nan,

        "width":
            np.nan,

        "height":
            np.nan,

        "estimated_container_duration_seconds":
            np.nan,
    }


def probe_video(
    video_path,
):
    """
    Reads technical metadata from one video container.

    The reported FPS and duration describe the exported
    video container, not necessarily the original clinical
    capture timeline.

    The source file is not modified.
    """

    video_path = Path(
        video_path
    )

    cap = cv2.VideoCapture(
        str(video_path)
    )

    try:
        if not cap.isOpened():
            return empty_video_probe()

        raw_fps = cap.get(
            cv2.CAP_PROP_FPS
        )

        raw_frame_count = cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )

        raw_width = cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )

        raw_height = cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )

        first_frame_readable, _ = (
            cap.read()
        )

    finally:
        cap.release()

    container_fps = (
        float(raw_fps)
        if (
            np.isfinite(raw_fps)
            and raw_fps > 0
        )
        else np.nan
    )


    container_reported_frame_count = (
        int(raw_frame_count)
        if (
            np.isfinite(raw_frame_count)
            and raw_frame_count > 0
            and float(raw_frame_count).is_integer()
        )
        else np.nan
    )


    width = (
        int(raw_width)
        if (
            np.isfinite(raw_width)
            and raw_width > 0
            and float(
                raw_width
            ).is_integer()
        )
        else np.nan
    )

    height = (
        int(raw_height)
        if (
            np.isfinite(raw_height)
            and raw_height > 0
            and float(
                raw_height
            ).is_integer()
        )
        else np.nan
    )

    estimated_container_duration_seconds = (
        float(container_reported_frame_count / container_fps)
        if (
            np.isfinite(container_reported_frame_count)
            and np.isfinite(container_fps)
        )
        else np.nan
    )


    return {
        "container_opened":
            True,

        "first_frame_readable":
            bool(
                first_frame_readable
            ),

        "container_fps":
            container_fps,

        "container_reported_frame_count":
            container_reported_frame_count,

        "width":
            width,

        "height":
            height,

        "estimated_container_duration_seconds":
            estimated_container_duration_seconds,
    }

In [30]:
# ------------------------------------------------------------------
# Build video manifest
# ------------------------------------------------------------------

VIDEO_MANIFEST_COLUMNS = [
    "video_key",
    "video_filename",
    "video_path",
    "video_relpath",
    "video_annotation_type",
    "video_size_bytes",
    *VIDEO_PROBE_COLUMNS,
]


def build_video_record(
    video_path,
    labelled_video_keys,
    storage_root,
):
    """
    Builds one structured manifest record for one
    physical video file.

    Technical video metadata is read without modifying
    the source file.
    """

    video_path = Path(
        video_path
    )

    storage_root = Path(
        storage_root
    )

    video_key = video_path.stem

    if not video_key:
        raise ValueError(
            "Cannot build video record from an empty "
            f"normalized video key: {video_path}"
        )

    storage_root = Path(
        storage_root
          ).resolve()


    video_path = (
          video_path.resolve()
        )


    try:
      video_relpath = (
          video_path
          .relative_to(
              storage_root
          )
          .as_posix()
      )

    except ValueError as error:
      raise ValueError(
          "Video path is outside the declared "
          "persistent storage root. "
          f"Video: {video_path}. "
          f"Storage root: {storage_root}."
      ) from error


    video_probe = probe_video(
        video_path
    )

    is_labelled_video = (
        video_key
        in labelled_video_keys
    )

    return {
        "video_key":
            video_key,

        "video_filename":
            video_path.name,

        "video_path":
            str(video_path),

        "video_relpath":
            video_relpath,

        "video_size_bytes": video_path.stat().st_size,

        "video_annotation_type": (
            "partially_labelled"
            if is_labelled_video
            else "fully_unlabelled"
        ),

        **video_probe,
    }


def build_video_manifest(
    video_files,
    labelled_video_keys,
    storage_root,
):
    """
    Builds a deterministic video-level manifest.

    One row represents one physical source video.
    """

    video_files = tuple(
        Path(video_path)
        for video_path in video_files
    )

    if not video_files:
        raise ValueError(
            "Cannot build a video manifest because no "
            "physical video files were provided."
        )

    duplicate_path_count = (
        len(video_files)
        - len(set(video_files))
    )

    if duplicate_path_count:
        raise ValueError(
            "The physical video inventory contains "
            f"{duplicate_path_count} duplicate paths."
        )

    paths_by_key = {}

    for path in video_files:
        key = path.stem  # same rule as build_video_record()
        paths_by_key.setdefault(key, []).append(path)

    duplicate_keys = {
        key: paths
        for key, paths in paths_by_key.items()
        if len(paths) > 1
    }

    if duplicate_keys:
        examples = {
            key: [str(path) for path in paths]
            for key, paths in list(duplicate_keys.items())[:10]
        }
        raise ValueError(
            f"Multiple physical videos have the same video_key: {examples}"
        )


    records = [
        build_video_record(
            video_path=video_path,
            labelled_video_keys=(
                labelled_video_keys
            ),
            storage_root=storage_root,
        )

        for video_path in tqdm(
            video_files,
            desc="Building video manifest",
        )
    ]

    return (
        pd.DataFrame.from_records(
            records,
            columns=(
                VIDEO_MANIFEST_COLUMNS
            ),
        )
        .sort_values(
            "video_key",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


labelled_video_keys = frozenset(
    df_clean[
        "video_key"
    ]
    .dropna()
    .astype("string")
    .str.strip()
    .loc[
        lambda values:
            values.ne("")
    ]
    .unique()
)


video_manifest = (
    build_video_manifest(
        video_files=video_files,
        labelled_video_keys=(
            labelled_video_keys
        ),
        storage_root=(
            CONFIG[
                "storage_root"
            ]
        ),
    )
)


manifest_path = DIRS["manifests_dir"] / "video_manifest.csv"
video_manifest.to_csv(manifest_path, index=False)

print("Video manifest saved to:", manifest_path)

print(
    "Video manifest rows:",
    f"{len(video_manifest):,}",
)


display(
    video_manifest.head()
)



Building video manifest:   0%|          | 0/117 [00:00<?, ?it/s]

Video manifest saved to: /content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/manifests/video_manifest.csv
Video manifest rows: 117


,video_key,video_filename,video_path,video_relpath,video_annotation_type,video_size_bytes,container_opened,first_frame_readable,container_fps,container_reported_frame_count,width,height,estimated_container_duration_seconds
0,04a78ef00c5245e0,04a78ef00c5245e0.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/labelled_vi...,partially_labelled,833894762,True,True,30.0,50986,336,336,1699.533333
1,0531325b64674948,0531325b64674948.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/labelled_vi...,partially_labelled,263695527,True,True,30.0,16151,336,336,538.366667
2,055bbbec392b4f3a,055bbbec392b4f3a.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/unlabelled_...,fully_unlabelled,736094508,True,True,30.0,36533,336,336,1217.766667
3,0728084c8da942d9,0728084c8da942d9.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/labelled_vi...,partially_labelled,798305289,True,True,30.0,48948,336,336,1631.600000
4,07c1fa15a20a4398,07c1fa15a20a4398.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/labelled_vi...,partially_labelled,603279074,True,True,30.0,38620,336,336,1287.333333


In [20]:
print(video_manifest.columns.tolist())
print(VIDEO_MANIFEST_COLUMNS)

['video_key', 'video_filename', 'video_path', 'video_relpath', 'video_annotation_type', 'video_size_bytes', 'container_opened', 'first_frame_readable', 'container_fps', 'container_reported_frame_count', 'width', 'height', 'estimated_container_duration_seconds']
['video_key', 'video_filename', 'video_path', 'video_relpath', 'video_annotation_type', 'video_size_bytes', 'container_opened', 'first_frame_readable', 'container_fps', 'container_reported_frame_count', 'width', 'height', 'estimated_container_duration_seconds']


### 10. Validate the video manifest data

In [ ]:
labelled_videos = video_manifest.loc[
    video_manifest["video_annotation_type"].eq("partially_labelled")
]

decode_records = []
audit_path = DIRS["results_dir"] / "labelled_video_decode_audit.csv"

for row in tqdm(
    labelled_videos.itertuples(index=False),
    total=len(labelled_videos),
    desc="Counting decoded frames",
):
    cap = cv2.VideoCapture(str(row.video_path))
    frames_read = 0
    stop_reason = "open_failed"
    error_message = None

    try:
        if cap.isOpened():
            stop_reason = "eof_or_read_failure"

            while True:
                readable, frame = cap.read()

                if not readable:
                    break

                if frame is None or frame.size == 0:
                    stop_reason = "empty_frame"
                    break

                frames_read += 1

    except cv2.error as error:
        stop_reason = "opencv_error"
        error_message = str(error)

    finally:
        cap.release()

    reported_frames = int(row.container_reported_frame_count)

    decode_records.append({
        "video_key": row.video_key,
        "container_reported_frames": reported_frames,
        "frames_read_before_stop": frames_read,
        "read_minus_reported": frames_read - reported_frames,
        "stop_reason": stop_reason,
        "error_message": error_message,
    })

    # Preserve progress after each video.
    decode_audit = pd.DataFrame(decode_records)
    decode_audit.to_csv(audit_path, index=False)

display(
    decode_audit.loc[
        decode_audit["read_minus_reported"].ne(0)
        | decode_audit["stop_reason"].ne("eof_or_read_failure")
    ]
)

print(
    "Frames read successfully:",
    f"{decode_audit['frames_read_before_stop'].sum():,}",
)
print("Published reference for this group: 1,955,675")
print("Audit saved to:", audit_path)

Counting decoded frames:   0%|          | 0/43 [00:00<?, ?it/s]

In [31]:
def validate_video_manifest(
    dataframe,
    labelled_video_keys,
    config,
):
    """
    Validates the complete physical video inventory,
    technical probe results, and consistency with
    labelled metadata.

    The input DataFrame is not modified.
    """

    required_columns = set(
        VIDEO_MANIFEST_COLUMNS
    )

    missing_columns = sorted(
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Video manifest is missing required columns: "
            f"{missing_columns}"
        )

    if dataframe.empty:
        raise ValueError(
            "Video manifest cannot be empty."
        )

    # --------------------------------------------------------------
    # Validate normalized video identifiers
    # --------------------------------------------------------------

    video_keys = (
        dataframe[
            "video_key"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_video_key_mask = (
        video_keys.isna()
        | video_keys.eq("")
    )

    if invalid_video_key_mask.any():
        raise ValueError(
            "Video manifest contains "
            f"{int(invalid_video_key_mask.sum())} "
            "missing or empty video keys."
        )

    duplicate_video_keys = (
        video_keys.loc[
            video_keys.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_video_keys:
        raise ValueError(
            "Multiple physical videos resolve to the same "
            f"video key: {duplicate_video_keys}"
        )

    # --------------------------------------------------------------
    # Validate physical and relative paths
    # --------------------------------------------------------------

    video_paths = (
        dataframe[
            "video_path"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_video_path_mask = (
        video_paths.isna()
        | video_paths.eq("")
    )

    if invalid_video_path_mask.any():
        raise ValueError(
            "Video manifest contains "
            f"{int(invalid_video_path_mask.sum())} "
            "missing video paths."
        )

    physical_file_exists = (
        video_paths.map(
            lambda value:
                Path(value).is_file()
        )
    )

    if not physical_file_exists.all():
        missing_files = (
            video_paths.loc[
                ~physical_file_exists
            ]
            .tolist()
        )

        raise FileNotFoundError(
            "Video manifest contains paths that do not "
            f"exist: {missing_files}"
        )

    duplicate_video_paths = (
        video_paths.loc[
            video_paths.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_video_paths:
        raise ValueError(
            "Duplicate physical video paths found: "
            f"{duplicate_video_paths}"
        )

    relative_paths = (
        dataframe[
            "video_relpath"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_relative_path_mask = (
        relative_paths.isna()
        | relative_paths.eq("")
        | relative_paths.map(
            lambda value:
                (
                    Path(value).is_absolute()
                    if pd.notna(value)
                    else True
                )
        )
    )

    if invalid_relative_path_mask.any():
        raise ValueError(
            "Video manifest contains invalid or absolute "
            "video-relative paths."
        )

    if relative_paths.duplicated().any():
        raise ValueError(
            "Video manifest contains duplicate "
            "video-relative paths."
        )

    # --------------------------------------------------------------
    # Validate complete physical inventory
    # --------------------------------------------------------------

    actual_total_videos = len(dataframe)

    minimum_video_files = int(
        config["dataset_validation"]["minimum_video_files"]
    )

    if actual_total_videos < minimum_video_files:
        raise ValueError(
            "Too few physical videos: "
            f"minimum {minimum_video_files}, "
            f"found {actual_total_videos}."
        )

    # --------------------------------------------------------------
    # Validate annotation categories and counts
    # --------------------------------------------------------------

    annotation_types = (
        dataframe[
            "video_annotation_type"
        ]
        .astype("string")
        .str.strip()
    )

    expected_annotation_types = {
        "partially_labelled",
        "fully_unlabelled",
    }

    invalid_annotation_mask = (
        annotation_types.isna()
        | annotation_types.eq("")
        | ~annotation_types.isin(
            expected_annotation_types
        )
    )

    if invalid_annotation_mask.any():
        invalid_annotation_types = (
            annotation_types.loc[
                invalid_annotation_mask
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            "Unexpected or missing video annotation "
            f"types: {invalid_annotation_types}"
        )

    annotation_counts = (
        annotation_types.value_counts()
    )

    actual_labelled_videos = int(
        annotation_counts.get(
            "partially_labelled",
            0,
        )
    )

    actual_unlabelled_videos = int(
        annotation_counts.get(
            "fully_unlabelled",
            0,
        )
    )

    expected_labelled_videos = len(frozenset(labelled_video_keys))
    expected_unlabelled_videos = actual_total_videos - expected_labelled_videos


    if (
        actual_labelled_videos
        != expected_labelled_videos
    ):
        raise ValueError(
            "Unexpected number of partially labelled "
            "videos: "
            f"expected {expected_labelled_videos}, "
            f"found {actual_labelled_videos}."
        )

    if (
        actual_unlabelled_videos
        != expected_unlabelled_videos
    ):
        raise ValueError(
            "Unexpected number of fully unlabelled "
            "videos: "
            f"expected {expected_unlabelled_videos}, "
            f"found {actual_unlabelled_videos}."
        )

    # --------------------------------------------------------------
    # Validate OpenCV probe results
    # --------------------------------------------------------------

    container_opened = (
        dataframe[
            "container_opened"
        ]
        .astype("boolean")
    )

    first_frame_readable = (
        dataframe[
            "first_frame_readable"
        ]
        .astype("boolean")
    )

    unreadable_container_mask = (
        container_opened.isna()
        | ~container_opened.fillna(False)
    )

    unreadable_first_frame_mask = (
        first_frame_readable.isna()
        | ~first_frame_readable.fillna(False)
    )

    if unreadable_container_mask.any():
        unreadable_videos = (
            dataframe.loc[
                unreadable_container_mask,
                "video_key",
            ]
            .tolist()
        )

        raise RuntimeError(
            "OpenCV could not open these videos: "
            f"{unreadable_videos}"
        )

    if unreadable_first_frame_mask.any():
        unreadable_videos = (
            dataframe.loc[
                unreadable_first_frame_mask,
                "video_key",
            ]
            .tolist()
        )

        raise RuntimeError(
            "OpenCV could not read the first frame of "
            f"these videos: {unreadable_videos}"
        )

    technical_metadata = (
        dataframe[
            [
                "container_fps",
                "container_reported_frame_count",
                "width",
                "height",
                "estimated_container_duration_seconds",
            ]
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    invalid_fps_mask = (
        technical_metadata[
            "container_fps"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "container_fps"
            ]
        )
        | technical_metadata[
            "container_fps"
        ].le(0)
    )

    invalid_frame_count_mask = (
        technical_metadata[
            "container_reported_frame_count"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "container_reported_frame_count"
            ]
        )
        | technical_metadata[
            "container_reported_frame_count"
        ].le(0)
        | technical_metadata[
            "container_reported_frame_count"
        ].mod(1).ne(0)
    )

    invalid_width_mask = (
        technical_metadata[
            "width"
        ].isna()
        | technical_metadata[
            "width"
        ].le(0)
        | technical_metadata[
            "width"
        ].mod(1).ne(0)
    )

    invalid_height_mask = (
        technical_metadata[
            "height"
        ].isna()
        | technical_metadata[
            "height"
        ].le(0)
        | technical_metadata[
            "height"
        ].mod(1).ne(0)
    )

    invalid_duration_mask = (
        technical_metadata[
            "estimated_container_duration_seconds"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "estimated_container_duration_seconds"
            ]
        )
        | technical_metadata[
            "estimated_container_duration_seconds"
        ].le(0)
    )

    invalid_technical_mask = (
        invalid_fps_mask
        | invalid_frame_count_mask
        | invalid_width_mask
        | invalid_height_mask
        | invalid_duration_mask
    )

    if invalid_technical_mask.any():
        invalid_videos = (
            dataframe.loc[
                invalid_technical_mask,
                "video_key",
            ]
            .tolist()
        )

        raise ValueError(
            "Invalid technical video metadata found for: "
            f"{invalid_videos}"
        )

    # --------------------------------------------------------------
    # Validate expected container FPS
    # --------------------------------------------------------------

    expected_container_fps = float(
        config[
            "expected_export_container_fps"
        ]
    )

    fps_matches_expected = np.isclose(
        technical_metadata[
            "container_fps"
        ].to_numpy(
            dtype=float
        ),
        expected_container_fps,
        rtol=0.0,
        atol=1e-3,
    )

    if not fps_matches_expected.all():
        fps_mismatches = (
            dataframe.loc[
                ~fps_matches_expected,
                [
                    "video_key",
                    "container_fps",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Unexpected exported-container FPS values. "
            f"Expected {expected_container_fps}: "
            f"{fps_mismatches}"
        )

    # --------------------------------------------------------------
    # Validate total extractable frame inventory
    # --------------------------------------------------------------

    actual_total_frames = int(
        technical_metadata[
            "container_reported_frame_count"
        ].sum()
    )

    expected_total_frames = int(
        config[
            "expected_total_extractable_frames"
        ]
    )

    if actual_total_frames != expected_total_frames:
        raise ValueError(
            "Unexpected total extractable frame count: "
            f"expected {expected_total_frames:,}, "
            f"found {actual_total_frames:,}."
        )

    # --------------------------------------------------------------
    # Cross-check labelled metadata against physical videos
    # --------------------------------------------------------------

    manifest_labelled_video_keys = frozenset(
        dataframe.loc[
            annotation_types.eq(
                "partially_labelled"
            ),
            "video_key",
        ]
    )

    metadata_labelled_video_keys = frozenset(
        labelled_video_keys
    )

    missing_physical_videos = sorted(
        metadata_labelled_video_keys
        - manifest_labelled_video_keys
    )

    unexpected_labelled_videos = sorted(
        manifest_labelled_video_keys
        - metadata_labelled_video_keys
    )

    if (
        missing_physical_videos
        or unexpected_labelled_videos
    ):
        raise ValueError(
            "Video manifest is inconsistent with metadata. "
            "Missing physical labelled videos: "
            f"{missing_physical_videos}. "
            "Unexpected labelled videos: "
            f"{unexpected_labelled_videos}."
        )

    return dataframe


video_manifest = (
    video_manifest
    .pipe(
        validate_video_manifest,
        labelled_video_keys=(
            labelled_video_keys
        ),
        config=CONFIG,
    )
)



video_counts = (
    video_manifest[
        "video_annotation_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "video_count"
    )
    .reset_index()
    .rename(
        columns={
            "video_annotation_type":
                "annotation_type"
        }
    )
)


video_inventory_summary = (
    pd.DataFrame.from_records(
        [
            {
                "total_videos":
                    len(video_manifest),

                "partially_labelled_videos":
                    int(
                        video_manifest[
                            "video_annotation_type"
                        ]
                        .eq(
                            "partially_labelled"
                        )
                        .sum()
                    ),

                "fully_unlabelled_videos":
                    int(
                        video_manifest[
                            "video_annotation_type"
                        ]
                        .eq(
                            "fully_unlabelled"
                        )
                        .sum()
                    ),

                "total_extractable_frames":
                    int(
                        video_manifest[
                            "frame_count"
                        ]
                        .sum()
                    ),

                "container_fps_min":
                    float(
                        video_manifest[
                            "container_fps"
                        ]
                        .min()
                    ),

                "container_fps_max":
                    float(
                        video_manifest[
                            "container_fps"
                        ]
                        .max()
                    ),
            }
        ]
    )
)


display(
    video_inventory_summary
)


display(
    video_counts
)

ValueError: Unexpected total extractable frame count: expected 4,741,504, found 4,765,114.